# HiPPO: Recurrent Memory with Optimal Polynomial Projections

HiPPO addresses the problem of storing the history of a sequence in a fixed-size memory.

In recurrent neural networks (RNNs), the hidden state is typically updated as

$$
h_t = f(h_{t-1}, x_t),
$$

where the model learns how to represent past information. However, there is no theoretical guarantee that this hidden state provides an optimal representation of the history.

The key idea behind HiPPO is to represent the past using a polynomial approximation,

$$
P(x) = a_0 + a_1x + a_2x^2 + \cdots + a_nx^n,
$$

and store only the polynomial coefficients instead of the entire history.

These coefficients provide the best polynomial approximation of the observed sequence under a chosen polynomial basis, allowing the model to summarize long histories with a fixed-size memory.

## Memory Update

1. Suppose we observe the following data:

    | Time | Observation |
    |------|-------------|
    | 1 | 2 |
    | 2 | 5 |
    | 3 | 4 |

The goal is to find the polynomial of degree 1 that best approximates the observed sequence.

$$
P(t)=a_0+a_1t
$$

2. Compute the polynomial coefficients.

    Let

    $$
    c(t)=[a_0(t),a_1(t)]
    $$

3. Now suppose a new observation arrives:

    | Time | Observation |
    |------|-------------|
    | 4 | 3 |

A straightforward approach would be to recompute the polynomial using all the observations.

### What does HiPPO do?

Instead of recomputing the polynomial from scratch, HiPPO directly updates the memory vector while preserving the best approximation of the observed sequence.

It achieves this by maintaining the coefficients of an optimal polynomial projection, which can be updated incrementally as new observations arrive.

## The Solution

Suppose the input is a continuous function,

$$
f(t).
$$

HiPPO demonstrates that the coefficients of the polynomial projection can be updated through a simple linear differential equation:

$$
\frac{dc(t)}{dt}=Ac(t)+Bf(t)
$$

Since the independent variable is time, it is common to use Newton's notation for derivatives:

$$
\dot{c}(t)=Ac(t)+Bf(t)
$$

| Term | Meaning |
|------|---------|
| $c(t)$ | Current memory (polynomial coefficients). |
| $\dot{c}(t)$ | How fast the memory changes. |
| $A$ | Updates the current memory as time evolves. |
| $Bf(t)$ | Incorporates the new observation into the memory. |

**Intuition**

- $A$ transforms the existing polynomial coefficients so that they remain the optimal representation as time advances.
- $B$ adds the contribution from the new observation.
- Observe that the equation does not depend on the whole polynomial, only on its coefficients.
- So, the problem addressed by HiPPO is to find matrices $A$ and $B$ such that the memory state always represents the optimal approximation of the history.

## Deriving the HiPPO Matrices

At this point, we have introduced the continuous update equation

$$
\dot{c}(t) = Ac(t) + Bf(t),
$$

where:

- $c(t)$ is the memory state (the polynomial coefficients),
- $A$ updates the existing memory,
- $B$ incorporates the new observation $f(t)$.

The next natural question is:

> **How do we choose the matrices $A$ and $B$?**

This is the main contribution of the HiPPO paper.

Unlike traditional polynomial fitting, HiPPO does not recompute the coefficients every time a new observation arrives. Instead, it seeks a pair of matrices $A$ and $B$ that allow the memory state to be updated **online**, while always preserving the optimal polynomial approximation of the observed history.

The derivation follows these steps:

1. **Define the optimal polynomial approximation** of the input history.
2. **Express the polynomial using an orthogonal basis** (Legendre polynomials in HiPPO-LegS).
3. **Differentiate the polynomial coefficients with respect to time**.
4. **Use the properties of the orthogonal basis to simplify the resulting expressions**.
5. **Obtain a linear differential equation of the form**

$$
\dot{c}(t)=Ac(t)+Bf(t),
$$

where the matrices $A$ and $B$ naturally emerge from the derivation.

The important point is that **HiPPO does not guess these matrices**. They are mathematically derived to guarantee that the memory state always represents the optimal projection of the input history.

### Intuition Behind Matrix $A$

Matrix $A$ defines how the memory evolves over time.

Its role is twofold:

- It gradually updates the existing memory as time passes.
- It redistributes information among the polynomial coefficients so that they continue to represent the history accurately.

For the Legendre measure, this derivation produces a lower triangular matrix. This structure allows information to flow from lower-order coefficients to higher-order ones while preserving the optimal representation.

Therefore, matrix $A$ should not be viewed as an arbitrary parameter, but rather as the mathematical rule that governs how the memory evolves over time.

Similarly, matrix $B$ determines how each new observation is incorporated into the memory state.

Together, $A$ and $B$ define a memory update rule that continuously maintains an optimal summary of the entire input history without recomputing the polynomial from scratch.

### Understanding the Structure of Matrix $A$

For the Legendre measure, the HiPPO derivation produces the following matrix:

$$
A_{nk}=
\begin{cases}
\sqrt{(2n+1)(2k+1)}, & n>k,\\
n+1, & n=k,\\
0, & n<k.
\end{cases}
$$

Although this expression may look intimidating at first, its structure is actually quite intuitive.

#### Upper triangular entries

When \(n < k\),

$$
A_{nk}=0.
$$

This means that coefficients of lower order are **not influenced** by higher-order coefficients.

Consequently, information only flows from lower-order polynomial coefficients toward higher-order ones, resulting in a **lower triangular matrix**.

---

#### Diagonal entries

When $n=k$,

$$
A_{nn}=n+1.
$$

These values determine how each coefficient evolves over time.

Higher-order coefficients have larger diagonal values, meaning that they change more rapidly than lower-order coefficients. Intuitively, higher-order polynomials represent finer details of the signal, which tend to disappear faster as time progresses.

---

#### Lower triangular entries

When $n>k$,

$$
A_{nk}=\sqrt{(2n+1)(2k+1)}.
$$

These terms couple different polynomial coefficients together.

Rather than evolving independently, each coefficient is influenced by lower-order coefficients, allowing information to propagate across the memory state while preserving the optimal polynomial representation.

The square-root expression is **not chosen manually**. It emerges naturally from the mathematical derivation using Legendre polynomials and their orthogonality properties.

---

Overall, matrix $A$ performs two important functions:

- It continuously updates the existing memory as time passes.
- It redistributes information among the polynomial coefficients to maintain the optimal approximation of the input history.

Therefore, matrix $A$ should not be viewed as an arbitrary matrix. Instead, it represents the mathematical rule that governs how the memory state evolves over time.

**Ecuaciones diferenciales**

https://www.youtube.com/watch?v=U7L2XmS7dl0

https://www.youtube.com/watch?v=_r9ch14tIro